In [1]:
import sys
print(sys.executable)
import pandas, folium
print("all good")

/home/cedricsalameh/bixi-project/venv/bin/python3
all good


In [2]:
import pandas as pd

path = "data/DonneesOuvertes2025_010203040506070809101112.csv"

dtypes = {
    "STARTSTATIONNAME": "category",
    "STARTSTATIONARRONDISSEMENT": "category",
    "ENDSTATIONNAME": "category",
    "ENDSTATIONARRONDISSEMENT": "category",
    "STARTSTATIONLATITUDE": "float32",
    "STARTSTATIONLONGITUDE": "float32",
    "ENDSTATIONLATITUDE": "float32",
    "ENDSTATIONLONGITUDE": "float32",
}

df = pd.read_csv(path, dtype=dtypes)
print(df.shape)
df.info(memory_usage="deep")

(14249363, 10)
<class 'pandas.DataFrame'>
RangeIndex: 14249363 entries, 0 to 14249362
Data columns (total 10 columns):
 #   Column                      Dtype   
---  ------                      -----   
 0   STARTSTATIONNAME            category
 1   STARTSTATIONARRONDISSEMENT  category
 2   STARTSTATIONLATITUDE        float32 
 3   STARTSTATIONLONGITUDE       float32 
 4   ENDSTATIONNAME              category
 5   ENDSTATIONARRONDISSEMENT    category
 6   ENDSTATIONLATITUDE          float32 
 7   ENDSTATIONLONGITUDE         float32 
 8   STARTTIMEMS                 int64   
 9   ENDTIMEMS                   float64 
dtypes: category(4), float32(4), float64(1), int64(1)
memory usage: 516.5 MB


In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "percent": missing_pct})

,missing,percent
STARTSTATIONNAME,0,0.00
STARTSTATIONARRONDISSEMENT,22,0.00
STARTSTATIONLATITUDE,0,0.00
STARTSTATIONLONGITUDE,0,0.00
ENDSTATIONNAME,51823,0.36
ENDSTATIONARRONDISSEMENT,52861,0.37
ENDSTATIONLATITUDE,51823,0.36
ENDSTATIONLONGITUDE,51823,0.36
STARTTIMEMS,0,0.00
ENDTIMEMS,51822,0.36


In [4]:
before = len(df)
df = df.dropna(subset=["ENDSTATIONNAME", "ENDTIMEMS"])
print(f"Dropped {before - len(df):,} trips ({(before - len(df)) / before:.2%})")

df["start"] = pd.to_datetime(df["STARTTIMEMS"], unit="ms", utc=True).dt.tz_convert("America/Montreal")
df["end"] = pd.to_datetime(df["ENDTIMEMS"], unit="ms", utc=True).dt.tz_convert("America/Montreal")
df["duration_min"] = (df["end"] - df["start"]).dt.total_seconds() / 60

df["duration_min"].describe()

Dropped 51,823 trips (0.36%)


count    1.419754e+07
mean     1.682043e+01
std      3.685056e+02
min      8.833333e-04
25%      6.067533e+00
50%      1.051867e+01
75%      1.795250e+01
max      3.712294e+05
Name: duration_min, dtype: float64

In [5]:
pd.set_option("display.float_format", "{:.2f}".format)

short = (df["duration_min"] < 1).sum()
long = (df["duration_min"] > 120).sum()
print(f"Under 1 min: {short:,} ({short / len(df):.2%})")
print(f"Over 2 hours: {long:,} ({long / len(df):.2%})")

Under 1 min: 184,996 (1.30%)
Over 2 hours: 45,440 (0.32%)


In [6]:
quick = df[df["duration_min"] < 1]
same = (quick["STARTSTATIONNAME"].astype(str) == quick["ENDSTATIONNAME"].astype(str)).mean()
print(f"{same:.1%} of sub-1-minute trips start and end at the same station")

95.2% of sub-1-minute trips start and end at the same station


In [7]:
before = len(df)
df = df[(df["duration_min"] >= 1) & (df["duration_min"] <= 120)].copy()
print(f"Dropped {before - len(df):,} trips, {len(df):,} remain")

df["hour"] = df["start"].dt.hour
df["end_hour"] = df["end"].dt.hour
df["month"] = df["start"].dt.month
df["weekday"] = df["start"].dt.dayofweek < 5

df = df.drop(columns=["STARTTIMEMS", "ENDTIMEMS"])
df.to_parquet("data/trips_clean.parquet")
print("Saved")

Dropped 230,436 trips, 13,967,104 remain
Saved
